# 06 — Interprétabilité du Modèle (SHAP)
**Aviation Safety Risk Prediction — NTSB + NOAA Dataset**

### Objectif
Expliquer les prédictions du modèle final (LightGBM) à l'aide des valeurs de Shapley (SHAP),
afin d'identifier quelles variables contribuent le plus à chaque prédiction, et en particulier
aux prédictions de la classe critique **FATL**.

### Pourquoi TreeExplainer ?
LightGBM est un modèle à base d'arbres. `shap.TreeExplainer` calcule les valeurs de Shapley
de manière exacte et rapide pour ce type de modèle, sans nécessiter d'échantillon de fond
(contrairement à `KernelExplainer`, plus lent et approximatif).

### Contenu
1. Chargement du modèle et des données
2. Calcul des valeurs SHAP (TreeExplainer, classe FATL)
3. Importance globale — bar plot
4. Distribution des contributions — beeswarm plot
5. Dependence plots — top features
6. Explications individuelles — waterfall plots (un cas FATL, un cas NONE)
7. Sauvegarde des artefacts


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, os, re, warnings
warnings.filterwarnings('ignore')

import shap

SEED       = 42
OUTPUT_DIR = 'outputs'
MODEL_DIR  = 'baseline_models'
SHAP_DIR   = 'shap_outputs'
os.makedirs(SHAP_DIR, exist_ok=True)

LABEL_MAP   = {0:'NONE', 1:'MINR', 2:'SERS', 3:'FATL'}
LABEL_NAMES = ['NONE', 'MINR', 'SERS', 'FATL']
FATL_IDX    = 3

print(f'shap version: {shap.__version__}')


## 1. Chargement du modèle et des données

In [ ]:
best_model = joblib.load(f'{MODEL_DIR}/best_model.pkl')
print(f'Modèle chargé : {type(best_model).__name__}')

X_train_raw = pd.read_csv(f'{OUTPUT_DIR}/X_train.csv')
X_test_raw  = pd.read_csv(f'{OUTPUT_DIR}/X_test.csv')
y_test      = pd.read_csv(f'{OUTPUT_DIR}/y_test.csv').squeeze()

try:
    feature_names = pd.read_csv(f'{OUTPUT_DIR}/feature_names.csv').squeeze().tolist()
except Exception:
    feature_names = X_train_raw.columns.tolist()

print(f'X_train shape : {X_train_raw.shape}')
print(f'X_test  shape : {X_test_raw.shape}')


In [ ]:
# Sanitize column names — même fonction que NB04, nécessaire si le modèle est LightGBM
def sanitize_lgb_columns(df):
    df = df.copy()
    new_cols = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in df.columns]
    seen = {}; deduped = []
    for col in new_cols:
        if col in seen: seen[col] += 1; deduped.append(f'{col}_{seen[col]}')
        else: seen[col] = 0; deduped.append(col)
    df.columns = deduped
    return df

is_lgb = 'LGBM' in type(best_model).__name__
if is_lgb:
    X_train_shap = sanitize_lgb_columns(X_train_raw)
    X_test_shap  = sanitize_lgb_columns(X_test_raw)
    print('LightGBM détecté — colonnes assainies pour SHAP')
else:
    X_train_shap = X_train_raw.copy()
    X_test_shap  = X_test_raw.copy()

# Échantillon aléatoire de 500 lignes du jeu de test pour accélérer le calcul SHAP
N_SAMPLE = min(500, len(X_test_shap))
sample_idx = np.random.RandomState(SEED).choice(len(X_test_shap), size=N_SAMPLE, replace=False)
X_sample = X_test_shap.iloc[sample_idx].reset_index(drop=True)
y_sample = y_test.iloc[sample_idx].reset_index(drop=True)

print(f'Échantillon SHAP : {X_sample.shape[0]} lignes (sur {len(X_test_shap)} au total)')


## 2. Calcul des valeurs SHAP
`TreeExplainer` est adapté aux modèles à base d'arbres (LightGBM, XGBoost, CatBoost, RandomForest).
Pour un problème multiclasse à 4 classes, `shap_values` renvoie un tableau de forme
`(n_samples, n_features, n_classes)` (API SHAP ≥ 0.42) — on isole ensuite la classe FATL (index 3),
classe prioritaire de ce projet.


In [ ]:
explainer = shap.TreeExplainer(best_model)
shap_values_raw = explainer(X_sample)

print(f'Type de sortie       : {type(shap_values_raw)}')
print(f'Shape des valeurs    : {shap_values_raw.values.shape}')
print(f'Shape attendue       : (n_samples={N_SAMPLE}, n_features={X_sample.shape[1]}, n_classes=4)')


In [ ]:
# Isoler les valeurs SHAP pour la classe FATL
if shap_values_raw.values.ndim == 3:
    shap_values_fatl = shap_values_raw.values[:, :, FATL_IDX]
    base_value_fatl   = shap_values_raw.base_values[:, FATL_IDX] if shap_values_raw.base_values.ndim > 1 \
                         else shap_values_raw.base_values
else:
    # Certains modèles binaires/anciennes versions renvoient un format 2D direct
    shap_values_fatl = shap_values_raw.values
    base_value_fatl   = shap_values_raw.base_values

print(f'shap_values_fatl shape : {shap_values_fatl.shape}')


## 3. Importance Globale — Bar Plot

In [ ]:
shap.summary_plot(
    shap_values_fatl, X_sample,
    plot_type='bar', max_display=20, show=False
)
plt.title('Importance globale des variables — classe FATL', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SHAP_DIR}/shap_bar_fatl.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Distribution des Contributions — Beeswarm Plot

In [ ]:
shap.summary_plot(
    shap_values_fatl, X_sample,
    max_display=20, show=False
)
plt.title('Distribution des contributions SHAP — classe FATL', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SHAP_DIR}/shap_beeswarm_fatl.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Dependence Plots — Top Features

In [ ]:
# Top 3 features par importance moyenne absolue
mean_abs_shap = np.abs(shap_values_fatl).mean(axis=0)
top3_idx = np.argsort(mean_abs_shap)[-3:][::-1]
top3_features = [X_sample.columns[i] for i in top3_idx]
print(f'Top 3 features : {top3_features}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, top3_features):
    shap.dependence_plot(
        feat, shap_values_fatl, X_sample,
        ax=ax, show=False
    )
    ax.set_title(feat, fontsize=10, fontweight='bold')

plt.suptitle('Dependence Plots — Top 3 Features (classe FATL)', fontsize=13, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(f'{SHAP_DIR}/shap_dependence_top3.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Explications Individuelles — Waterfall Plots
On illustre le raisonnement du modèle sur deux cas concrets : un accident réellement classé FATL,
et un accident réellement classé NONE.


In [ ]:
# Trouver un exemple FATL et un exemple NONE dans l'échantillon
idx_fatl_example = np.where(y_sample.values == 3)[0]
idx_none_example = np.where(y_sample.values == 0)[0]

print(f'Exemples FATL disponibles : {len(idx_fatl_example)}')
print(f'Exemples NONE disponibles : {len(idx_none_example)}')


In [ ]:
if len(idx_fatl_example) > 0:
    i = idx_fatl_example[0]
    explanation_fatl = shap.Explanation(
        values=shap_values_fatl[i],
        base_values=base_value_fatl[i] if np.ndim(base_value_fatl) > 0 else base_value_fatl,
        data=X_sample.iloc[i].values,
        feature_names=X_sample.columns.tolist()
    )
    shap.plots.waterfall(explanation_fatl, max_display=15, show=False)
    plt.title('Explication individuelle — exemple réel FATL', fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{SHAP_DIR}/shap_waterfall_fatl_example.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
if len(idx_none_example) > 0:
    i = idx_none_example[0]
    explanation_none = shap.Explanation(
        values=shap_values_fatl[i],
        base_values=base_value_fatl[i] if np.ndim(base_value_fatl) > 0 else base_value_fatl,
        data=X_sample.iloc[i].values,
        feature_names=X_sample.columns.tolist()
    )
    shap.plots.waterfall(explanation_none, max_display=15, show=False)
    plt.title('Explication individuelle — exemple réel NONE\n(contribution à la probabilité FATL)',
               fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{SHAP_DIR}/shap_waterfall_none_example.png', dpi=150, bbox_inches='tight')
    plt.show()


## 7. Tableau Récapitulatif des Variables les Plus Influentes

In [ ]:
importance_df = pd.DataFrame({
    'feature': X_sample.columns,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

importance_df.to_csv(f'{SHAP_DIR}/shap_feature_importance.csv', index=False)
print('Top 15 variables les plus influentes pour la classe FATL :')
print(importance_df.head(15).to_string(index=False))


## 8. Sauvegarde des Artefacts

In [ ]:
joblib.dump(explainer, f'{SHAP_DIR}/shap_explainer.pkl')
np.save(f'{SHAP_DIR}/shap_values_fatl.npy', shap_values_fatl)

print('=' * 60)
print('  INTERPRÉTABILITÉ SHAP — RÉSUMÉ')
print('=' * 60)
print(f'  Modèle expliqué        : {type(best_model).__name__}')
print(f'  Méthode                : shap.TreeExplainer')
print(f'  Classe analysée        : FATL')
print(f'  Échantillon            : {N_SAMPLE} observations du jeu de test')
print(f'  Variable la plus influente : {importance_df.iloc[0]["feature"]}')
print(f'\n  Artefacts sauvegardés dans {SHAP_DIR}/ :')
for f in sorted(os.listdir(SHAP_DIR)):
    print(f'    {f}')

print(f'\n  → Ces résultats sont intégrés au rapport (Chapitre 4 — Réalisation, Chapitre 5 — Tests et Résultats)')
